# RNN/LSTM sequence modeling: 직접 구현하고 같은 조건에서 비교하기

이 과제는 앞 수업의 RNN/LSTM 설명을 한 흐름으로 연결합니다. 지연된 첫 토큰을 기억해야 하는 이진 분류 task에서 직접 만든 RNN cell, LSTM cell, sequence classifier, 마지막 토큰 baseline을 같은 데이터와 학습 조건으로 비교합니다.

각 번호는 `구현 → fixture → check_e##() → 해석` 순서입니다. 처음에는 모든 fixture와 checker가 TODO 경계에서 멈추는 것이 정상입니다. 구현을 완료한 뒤 위에서 아래로 다시 실행하고, 마지막 비교 결과와 실패 원인을 자신의 말로 기록하세요.


In [26]:
import torch
from torch import nn
import torch.nn.functional as F


## 1. 지연 정보 dataset 만들기

### 목적

마지막 입력만 보면 답을 알 수 없고, 첫 토큰을 끝까지 보존해야만 label을 맞힐 수 있는 가장 작은 sequence task를 만듭니다.

### 이번 계약

- Create fixed delayed-information train and evaluation splits whose first token determines a sequence-level binary label and whose later tokens are zero.
- `examples`와 `delay`는 모두 1 이상이어야 하며, 그렇지 않으면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

예를 들어 delay가 3이면 입력 한 행은 `[첫 bit, 0, 0, 0]`이고 label은 첫 bit입니다. 마지막 토큰은 언제나 0이므로 마지막 토큰만 읽는 baseline은 이 task의 단서를 볼 수 없습니다.

### 해보기

TODO에서 첫 bit를 첫 위치에 넣고, 그 같은 bit를 `long` label로 만드세요. 나머지 위치는 제공된 0을 유지하세요.

<details><summary>힌트 1</summary>

`torch.zeros((examples, delay + 1, 1))`의 첫 sequence 위치만 바꾸면 됩니다.

</details>

<details><summary>힌트 2</summary>

분류 label은 class index이므로 `long` dtype의 `(examples,)` shape로 만드세요.

</details>

### 실행과 확인

fixture와 `check_e01()`은 shape, 첫 토큰과 label의 연결, 한 행 경계, 잘못된 크기를 확인합니다.


In [27]:
def make_delayed_dataset(*, examples: int, delay: int, seed: int) -> tuple[torch.Tensor, torch.Tensor]:
    if examples < 1 or delay < 1:
        raise ValueError("examples and delay must be positive")
    generator = torch.Generator().manual_seed(seed)
    first_bits = torch.randint(0, 2, (examples,), generator=generator)
    inputs = torch.zeros((examples, delay + 1, 1), dtype=torch.float32)
    # Local contract
    # - inputs의 축은 (batch, sequence, feature)이며 shape는 (examples, delay + 1, 1)입니다.
    # - first_bits는 각 예시의 첫 위치에 놓일 bit이고 뒤 위치의 입력은 0으로 남습니다.
    # - labels는 같은 bit를 담는 shape (batch,)의 torch.long Tensor입니다.
    # - 반환값은 (inputs, labels)이며 inputs dtype은 torch.float32입니다.
    # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
    inputs[:, 0, 0] = first_bits
    labels = torch.asarray(first_bits)
    # 학습자 편집 구간 끝

    return inputs, labels

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e01()


In [28]:
demo_inputs, demo_labels = make_delayed_dataset(examples=4, delay=3, seed=7)
print("inputs:", demo_inputs.squeeze(-1))
print("labels:", demo_labels)


inputs: tensor([[1., 0., 0., 0.],
        [0., 0., 0., 0.],
        [1., 0., 0., 0.],
        [0., 0., 0., 0.]])
labels: tensor([1, 0, 1, 0])


In [29]:
def check_e01() -> None:
    inputs, labels = make_delayed_dataset(examples=4, delay=3, seed=7)
    torch.testing.assert_close(tuple(inputs.shape), (4, 4, 1))
    torch.testing.assert_close(labels, inputs[:, 0, 0].long())
    one_input, one_label = make_delayed_dataset(examples=1, delay=1, seed=0)
    torch.testing.assert_close((tuple(one_input.shape), tuple(one_label.shape)), ((1, 2, 1), (1,)))
    try:
        make_delayed_dataset(examples=0, delay=1, seed=0)
    except ValueError:
        pass
    else:
        raise AssertionError("nonpositive examples must fail")


check_e01()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 2. 수동 vanilla RNN cell 구현하기

### 목적

`nn.RNN`에 맡기지 않고 현재 입력과 직전 hidden state가 한 cell에서 어떻게 합쳐지는지 직접 구현합니다.

### 이번 계약

- `ManualRNNCell.forward`는 `(B, I)` 입력과 `(B, H)` 이전 hidden state에서 `tanh(input_to_hidden(x) + hidden_to_hidden(h_prev))`를 반환해야 합니다.
- 입력 또는 이전 hidden state의 batch 축과 feature 축이 cell 설정과 다르면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

batch가 2이고 input size가 1, hidden size가 3이면 현재 입력은 `(2, 1)`, 직전 hidden state는 `(2, 3)`, 다음 hidden state도 `(2, 3)`입니다.

### 해보기

TODO에서 두 affine 결과를 더한 뒤 tanh를 적용하세요. `__init__`의 두 `nn.Linear`와 forward의 shape 검사는 유지하세요.

<details><summary>힌트 1</summary>

두 `nn.Linear`는 각각 `(B, I)`와 `(B, H)`를 `(B, H)`로 바꿉니다.

</details>

<details><summary>힌트 2</summary>

더한 결과 전체에 `torch.tanh`를 한 번 적용해야 다음 hidden state가 됩니다.

</details>

### 실행과 확인

fixture와 `check_e02()`는 수식과 동일한 output, batch 1 경계, 잘못된 hidden shape를 확인합니다.


In [30]:
class ManualRNNCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.input_to_hidden = nn.Linear(input_size, hidden_size)
        self.hidden_to_hidden = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        expected_x = (x.ndim == 2 and x.shape[1] == self.input_size)
        expected_h = h_prev.shape == (x.shape[0], self.hidden_size)
        if not expected_x or not expected_h:
            raise ValueError("x and h_prev must match the cell shape contract")

        # Local contract
        # - x는 현재 위치의 입력 (batch, input_size)입니다.
        # - h_prev는 이전 위치에서 넘어온 hidden state (batch, hidden_size)입니다.
        # - 두 projection의 결과 shape는 모두 (batch, hidden_size)여야 합니다.
        # - 두 결과를 결합한 뒤 tanh를 적용하고 같은 shape의 next_h를 반환합니다.
        # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
        next_h = torch.tanh(
            self.input_to_hidden(x) + self.hidden_to_hidden(h_prev)
        )
        # 학습자 편집 구간 끝

        return next_h

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e02()


In [31]:
torch.manual_seed(1)
rnn_cell = ManualRNNCell(input_size=1, hidden_size=3)
demo_h = rnn_cell(torch.tensor([[1.0], [0.0]]), torch.zeros((2, 3)))
print("next hidden shape:", tuple(demo_h.shape))


next hidden shape: (2, 3)


In [32]:
def check_e02() -> None:
    torch.manual_seed(2)
    cell = ManualRNNCell(input_size=1, hidden_size=2)
    x = torch.tensor([[1.0], [-1.0]])
    h_prev = torch.zeros((2, 2))
    expected = torch.tanh(cell.input_to_hidden(x) + cell.hidden_to_hidden(h_prev))
    torch.testing.assert_close(cell(x, h_prev), expected)
    torch.testing.assert_close(tuple(cell(torch.zeros((1, 1)), torch.zeros((1, 2))).shape), (1, 2))
    try:
        cell(torch.zeros((2, 1)), torch.zeros((2, 3)))
    except ValueError:
        pass
    else:
        raise AssertionError("wrong hidden shape must fail")


check_e02()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 3. 수동 LSTM cell 구현하기

### 목적

LSTM이 hidden state만 이어받는 것이 아니라 별도 cell state를 gate로 조절하는 것을 실제 Tensor 연산으로 만듭니다.

### 이번 계약

- `ManualLSTMCell.forward`는 input, forget, candidate, output gate를 만들고 `next_c = forget * c_prev + input_gate * candidate`, `next_h = output_gate * tanh(next_c)`를 반환해야 합니다.
- 입력, 이전 hidden state, 이전 cell state의 batch 축과 hidden feature 축이 cell 설정과 다르면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

hidden size가 2이면 gate pre-activation의 마지막 축은 8이고, 이를 hidden size 2짜리 네 묶음으로 나눕니다. `c_prev`와 `h_prev`는 둘 다 `(B, 2)`이지만 역할은 다릅니다.

### 해보기

TODO에서 gate pre-activation을 네 묶음으로 나누고 sigmoid/tanh를 적용한 뒤 `next_c`, `next_h`를 계산하세요.

<details><summary>힌트 1</summary>

`gates.chunk(4, dim=1)`의 순서는 이 과제에서 input, forget, candidate, output입니다.

</details>

<details><summary>힌트 2</summary>

input/forget/output은 sigmoid, candidate는 tanh를 쓰고 cell update에는 이전 `c_prev`가 직접 들어갑니다.

</details>

### 실행과 확인

fixture와 `check_e03()`는 두 state의 shape, batch 1 경계, 잘못된 cell-state shape를 확인합니다.


In [33]:
class ManualLSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int) -> None:
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.input_to_gates = nn.Linear(input_size, 4 * hidden_size)
        self.hidden_to_gates = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def forward(
        self, x: torch.Tensor, h_prev: torch.Tensor, c_prev: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        expected_x = (x.ndim == 2 and x.shape[1] == self.input_size)
        expected_state = (x.shape[0], self.hidden_size)
        if not expected_x or h_prev.shape != expected_state or c_prev.shape != expected_state:
            raise ValueError("x, h_prev, and c_prev must match the cell shape contract")

        # Local contract
        # - x는 현재 입력, h_prev는 이전 hidden, c_prev는 이전 내부 cell memory입니다.
        # - 세 입력/state shape는 각각 (batch, input_size), (batch, hidden_size), (batch, hidden_size)입니다.
        # - 두 projection을 합친 마지막 축을 input, forget, candidate, output 순서로 나눕니다.
        # - 세 gate는 sigmoid로 비율을 만들고 candidate는 tanh로 후보 내용을 만듭니다.
        # - cell memory를 먼저 갱신한 뒤 그중 노출할 hidden state를 만듭니다.
        # - 반환 순서는 (next_h, next_c)이며 둘 다 shape는 (batch, hidden_size)입니다.
        # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
        gates = self.input_to_gates(x) + self.hidden_to_gates(h_prev)
        raw_i, raw_f, raw_g, raw_o = gates.chunk(4, dim=1)

        input_gate = torch.sigmoid(raw_i)
        forget_gate = torch.sigmoid(raw_f)
        candidate = torch.tanh(raw_g)
        output_gate = torch.sigmoid(raw_o)

        next_c = forget_gate * c_prev + input_gate * candidate
        next_h = output_gate * torch.tanh(next_c)
        # 학습자 편집 구간 끝

        return next_h, next_c

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e03()


In [34]:
torch.manual_seed(3)
lstm_cell = ManualLSTMCell(input_size=1, hidden_size=2)
demo_h, demo_c = lstm_cell(torch.ones((2, 1)), torch.zeros((2, 2)), torch.zeros((2, 2)))
print("hidden and cell shapes:", tuple(demo_h.shape), tuple(demo_c.shape))


hidden and cell shapes: (2, 2) (2, 2)


In [35]:
def check_e03() -> None:
    torch.manual_seed(4)
    cell = ManualLSTMCell(input_size=1, hidden_size=2)
    x = torch.tensor([[1.0], [-1.0]])
    h_prev = torch.tensor([[0.2, -0.1], [0.0, 0.3]])
    c_prev = torch.tensor([[0.4, -0.2], [0.1, 0.5]])
    next_h, next_c = cell(x, h_prev, c_prev)
    gates = cell.input_to_gates(x) + cell.hidden_to_gates(h_prev)
    raw_i, raw_f, raw_g, raw_o = gates.chunk(4, dim=1)
    expected_c = torch.sigmoid(raw_f) * c_prev + torch.sigmoid(raw_i) * torch.tanh(raw_g)
    expected_h = torch.sigmoid(raw_o) * torch.tanh(expected_c)
    torch.testing.assert_close(next_c, expected_c)
    torch.testing.assert_close(next_h, expected_h)
    one_h, one_c = cell(torch.ones((1, 1)), torch.zeros((1, 2)), torch.zeros((1, 2)))
    torch.testing.assert_close((tuple(one_h.shape), tuple(one_c.shape)), ((1, 2), (1, 2)))
    try:
        cell(torch.ones((2, 1)), torch.zeros((2, 2)), torch.zeros((2, 3)))
    except ValueError:
        pass
    else:
        raise AssertionError("wrong cell-state shape must fail")


check_e03()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 4. sequence classifier와 마지막 토큰 baseline 연결하기

### 목적

cell 하나를 sequence 전체에 반복 적용하고, 문장 하나당 하나의 label을 내기 위해 마지막 hidden state를 readout에 연결합니다.

### 이번 계약

- Implement `ManualRNNCell`, `ManualLSTMCell`, and `SequenceClassifier` as reusable `nn.Module` components with explicit `forward` methods; `SequenceClassifier` stores its recurrent component in `cell`.
- `SequenceClassifier.forward`는 모든 token을 시간 순서대로 처리하고 마지막 hidden state를 `readout`에 넣어 `(B, 2)` logits를 반환해야 합니다.
- `kind`는 `rnn` 또는 `lstm`만 허용하며, 다른 값이면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

입력이 `(B, S, 1)`이면 각 시점 cell 입력은 `(B, 1)`입니다. sequence classifier는 마지막 `(B, H)` hidden state를 선형 readout에 넣어 `(B, 2)` logits를 만듭니다.

### 해보기

TODO에서 RNN/LSTM 분기를 포함한 time-step loop, 마지막 hidden state 선택, readout을 완성하세요. 제공된 `LastTokenLinearBaseline`은 마지막 입력만 읽는 비교 기준입니다.

<details><summary>힌트 1</summary>

LSTM cell은 `(h, c)`를 돌려주므로 다음 step에 둘 다 넘기고, RNN cell은 `h` 하나만 갱신합니다.

</details>

<details><summary>힌트 2</summary>

sequence-level 분류이므로 모든 step output을 쌓을 필요 없이 마지막 `h`를 `self.readout`에 넣으면 됩니다.

</details>

### 실행과 확인

fixture와 `check_e04()`는 RNN과 LSTM의 time-step state update, 마지막 hidden readout, batch 1 baseline, 지원하지 않는 kind를 확인합니다.


In [36]:
class SequenceClassifier(nn.Module):
    def __init__(self, kind: str, input_size: int, hidden_size: int, num_classes: int = 2) -> None:
        super().__init__()
        if kind not in {"rnn", "lstm"}:
            raise ValueError("kind must be rnn or lstm")
        self.kind = kind
        self.hidden_size = hidden_size
        self.cell = ManualRNNCell(input_size, hidden_size) if kind == "rnn" else ManualLSTMCell(input_size, hidden_size)
        self.readout = nn.Linear(hidden_size, num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if inputs.ndim != 3 or inputs.shape[2] != 1:
            raise ValueError("inputs must have shape (batch, steps, 1)")
        batch_size = inputs.shape[0]
        h = torch.zeros((batch_size, self.hidden_size), dtype=inputs.dtype, device=inputs.device)
        c = torch.zeros_like(h)



        # Local contract
        # - inputs의 축은 (batch, steps, input)입니다.
        # - 시점 loop에서는 한 번에 현재 위치의 (batch, input) Tensor를 cell에 전달합니다.
        # - RNN은 h만, LSTM은 h와 c를 다음 위치로 이어 갑니다.
        # - 마지막 위치까지 처리한 h를 readout에 넣습니다.
        # - 반환 logits의 shape는 (batch, num_classes)입니다.
        # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
        for t in range(inputs.shape[1]):
            x_t = inputs[:, t, :]

            if self.kind == "rnn":
                h = self.cell(x_t, h)
            else:
                h, c = self.cell(x_t, h, c)

        logits = self.readout(h)
        # 학습자 편집 구간 끝

        return logits


class LastTokenLinearBaseline(nn.Module):
    def __init__(self, input_size: int = 1, num_classes: int = 2) -> None:
        super().__init__()
        self.readout = nn.Linear(input_size, num_classes)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.readout(inputs[:, -1, :])

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e04()


In [37]:
torch.manual_seed(5)
classifier = SequenceClassifier("rnn", input_size=1, hidden_size=4)
demo_logits = classifier(torch.zeros((3, 5, 1)))
print("sequence logits shape:", tuple(demo_logits.shape))


sequence logits shape: (3, 2)


In [38]:
def check_e04() -> None:
    model = SequenceClassifier("rnn", input_size=1, hidden_size=1)
    with torch.no_grad():
        model.cell.input_to_hidden.weight.fill_(1.0)
        model.cell.input_to_hidden.bias.zero_()
        model.cell.hidden_to_hidden.weight.fill_(1.0)
        model.readout.weight.copy_(torch.tensor([[1.0], [-1.0]]))
        model.readout.bias.zero_()
    rnn_inputs = torch.tensor([[[1.0], [0.0], [0.0]]])
    expected_h = torch.zeros((1, 1))
    for token in rnn_inputs.unbind(dim=1):
        expected_h = torch.tanh(model.cell.input_to_hidden(token) + model.cell.hidden_to_hidden(expected_h))
    expected_logits = model.readout(expected_h)
    torch.testing.assert_close(isinstance(model.cell, ManualRNNCell), True)
    torch.testing.assert_close(model(rnn_inputs), expected_logits)

    lstm_model = SequenceClassifier("lstm", input_size=1, hidden_size=1)
    lstm_inputs = torch.tensor([[[1.0], [0.0]]])
    expected_h = torch.zeros((1, 1))
    expected_c = torch.zeros((1, 1))
    for token in lstm_inputs.unbind(dim=1):
        expected_h, expected_c = lstm_model.cell(token, expected_h, expected_c)
    torch.testing.assert_close(lstm_model(lstm_inputs), lstm_model.readout(expected_h))

    baseline = LastTokenLinearBaseline()
    torch.testing.assert_close(tuple(baseline(torch.zeros((1, 2, 1))).shape), (1, 2))
    try:
        SequenceClassifier("gru", input_size=1, hidden_size=3)
    except ValueError:
        pass
    else:
        raise AssertionError("unsupported kind must fail")


check_e04()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 5. sequence-level cross-entropy 만들기

### 목적

classifier가 낸 logits와 label shape를 명확히 맞추고, training loop가 호출할 loss 함수를 분리합니다.

### 이번 계약

- Compute cross-entropy from sequence-level logits with shape `(B, 2)` and labels with shape `(B,)`.
- logits가 rank 2가 아니거나 labels가 rank 1이 아니거나 batch 크기가 다르면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

batch 3의 문장 분류라면 logits는 `(3, 2)`, labels는 `(3,)`입니다. `(B, S, 2)` 전체 output을 그대로 넣는 token tagging loss와는 다른 계약입니다.

### 해보기

TODO에서 제공된 shape 검사를 통과한 logits와 labels에 cross entropy를 계산하세요.

<details><summary>힌트 1</summary>

이 함수의 logits는 이미 마지막 hidden state에서 나온 sequence-level scores입니다.

</details>

<details><summary>힌트 2</summary>

PyTorch의 cross entropy는 class index label에 softmax를 따로 적용하지 않은 logits를 받습니다.

</details>

### 실행과 확인

fixture와 `check_e05()`는 batch 3 normal case, batch 1 edge case, label rank failure를 확인합니다.


In [39]:
def sequence_cross_entropy(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    same_batch = logits.ndim == 2 and labels.ndim == 1 and logits.shape[0] == labels.shape[0]
    if not same_batch:
        raise ValueError("logits must be (B, C) and labels must be (B,)")

    # Local contract
    # - logits shape는 (batch, classes), labels shape는 (batch,)입니다.
    # - labels dtype은 torch.long이며 각 값은 정답 class index입니다.
    # - cross-entropy에 넣기 전에 softmax를 적용하지 않습니다.
    # - 반환값 loss는 원소 하나인 scalar Tensor입니다.
    # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
    # TODO: sequence-level logits와 labels의 cross-entropy를 계산하세요.

    loss = F.cross_entropy(logits, labels)

    # 학습자 편집 구간 끝

    return loss

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e05()


In [40]:
demo_loss = sequence_cross_entropy(
    torch.tensor([[2.0, -1.0], [-1.0, 2.0]]), torch.tensor([0, 1])
)
print("sequence loss:", float(demo_loss))


sequence loss: 0.04858732968568802


In [41]:
def check_e05() -> None:
    logits = torch.tensor([[2.0, -1.0], [-1.0, 2.0]])
    labels = torch.tensor([0, 1])
    torch.testing.assert_close(sequence_cross_entropy(logits, labels), F.cross_entropy(logits, labels))
    torch.testing.assert_close(
        sequence_cross_entropy(torch.tensor([[0.0, 1.0]]), torch.tensor([1])),
        F.cross_entropy(torch.tensor([[0.0, 1.0]]), torch.tensor([1])),
    )
    try:
        sequence_cross_entropy(logits, labels.unsqueeze(1))
    except ValueError:
        pass
    else:
        raise AssertionError("rank-2 labels must fail")


check_e05()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 6. 학습 loop와 train-loss 관찰 만들기

### 목적

파라미터는 forward가 아니라 optimizer의 `step`에서 바뀐다는 사실을 실제 RNN/LSTM 학습 loop로 연결합니다.

### 이번 계약

- Run a learner-owned `zero_grad` → forward → loss → backward → `step` loop for a fixed number of epochs and return the scalar loss history.
- `epochs`는 1 이상이어야 하며, 그렇지 않으면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

한 epoch의 순서는 `zero_grad`, forward, loss, backward, step입니다. `zero_grad`를 빼면 이전 batch gradient가 누적되고, `step`을 빼면 파라미터가 바뀌지 않습니다.

### 해보기

TODO에서 `backward` 뒤 parameter를 실제로 바꾸는 optimizer step을 호출하세요. 나머지 full-batch loop와 scalar loss 기록은 제공되어 있어, update 위치와 파라미터 변경을 분명하게 관찰할 수 있습니다.

<details><summary>힌트 1</summary>

loss scalar를 history에 저장할 때는 graph를 보관하지 않도록 Python float로 바꾸세요.

</details>

<details><summary>힌트 2</summary>

`optimizer.step()`은 `loss.backward()` 뒤에 있어야 gradient를 사용해 parameter를 갱신합니다.

</details>

### 실행과 확인

fixture는 baseline의 train loss history를 출력합니다. `check_e06()`은 한 update history와 epoch 경계가 올바른지 확인합니다.


In [42]:
def fit_model(
    model: nn.Module,
    train_inputs: torch.Tensor,
    train_labels: torch.Tensor,
    *,
    epochs: int,
    learning_rate: float,
) -> list[float]:
    if epochs < 1:
        raise ValueError("epochs must be positive")
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history: list[float] = []
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(train_inputs)
        loss = sequence_cross_entropy(logits, train_labels)
        loss.backward()
        # Local contract
        # - 한 epoch의 고정 순서는 zero_grad → forward → loss → backward → step입니다.
        # - zero_grad는 이전 gradient를 비우고 forward는 현재 parameter로 logits를 만듭니다.
        # - loss는 예측 오차를 scalar로 만들고 backward는 parameter.grad를 채웁니다.
        # - 마지막에는 optimizer의 갱신 메서드를 호출하며 메서드 자체를 덮어쓰지 않습니다.
        # - history에는 갱신 직전 이 epoch에서 관찰한 loss 값이 추가됩니다.
        # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
        # TODO: backward 뒤에 parameter를 갱신하는 optimizer step을 실행하세요.
        optimizer.step()
        # 학습자 편집 구간 끝
        history.append(float(loss.detach()))

    return history

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e06()


In [43]:
train_inputs, train_labels = make_delayed_dataset(examples=32, delay=6, seed=11)
train_baseline = LastTokenLinearBaseline()
train_history = fit_model(train_baseline, train_inputs, train_labels, epochs=3, learning_rate=0.05)
print("train loss history:", train_history)


train loss history: [0.7312860488891602, 0.7174255847930908, 0.7061092853546143]


In [44]:
def check_e06() -> None:
    inputs, labels = make_delayed_dataset(examples=8, delay=2, seed=12)
    model = LastTokenLinearBaseline()
    before = [parameter.detach().clone() for parameter in model.parameters()]
    history = fit_model(model, inputs, labels, epochs=1, learning_rate=0.05)
    torch.testing.assert_close(len(history), 1)
    torch.testing.assert_close(torch.isfinite(torch.tensor(history)).all(), torch.tensor(True))
    changed = any(not torch.equal(old, new) for old, new in zip(before, model.parameters()))
    torch.testing.assert_close(changed, True)
    try:
        fit_model(model, inputs, labels, epochs=0, learning_rate=0.05)
    except ValueError:
        pass
    else:
        raise AssertionError("zero epochs must fail")


check_e06()


### 메모

선택 메모입니다. 이 메모는 실습 완료 조건이 아닙니다.


## 7. 동일 조건 평가·비교와 실패 진단

### 목적

RNN, LSTM, 마지막 토큰 baseline을 같은 generated data, split, hidden size, readout, optimizer, learning rate, epoch budget으로 학습하고 held-out accuracy를 해석합니다.

### 이번 계약

- Evaluate every model on the same held-out split under `torch.no_grad()` and return accuracy without updating parameters.
- 관찰한 train-loss와 held-out comparison을 바탕으로 RNN·LSTM·baseline의 결과를 적고, 매 step `detach`, 마지막 hidden이 아닌 잘못된 readout, logits/labels shape 불일치 중 하나가 왜 실패를 만드는지 진단해야 합니다.
- 평가 labels가 rank 1이 아니거나 inputs와 batch 크기가 다르면 `ValueError`를 발생시켜야 합니다.

### 작은 추적

같은 held-out split에서 accuracy가 달라져야 비교가 의미 있습니다. 마지막 토큰 baseline의 입력은 항상 0이므로, 높은 accuracy가 나와도 첫 토큰 정보를 사용했다는 결론을 내릴 수 없습니다.

### 해보기

TODO에서 `model.eval()`과 `torch.no_grad()` 아래 logits, argmax prediction, accuracy, count를 조립하세요. fixture는 세 모델을 같은 조건으로 학습·평가합니다.

<details><summary>힌트 1</summary>

evaluation에서는 `backward`나 `step`이 없어야 하고, accuracy는 `prediction == labels`의 평균입니다.

</details>

<details><summary>힌트 2</summary>

fixture의 세 모델은 같은 seed와 dataset split을 쓰며 RNN/LSTM만 hidden size와 readout 구조를 공유합니다. 한 번의 작은 실험은 일반화 증명이 아니라 관찰입니다.

</details>

### 실행과 확인

fixture 출력은 나중에 해석할 실제 comparison result입니다. `check_e07()`은 완벽한 제공 model의 accuracy, batch 1 edge, 잘못된 label shape를 확인합니다.


In [45]:
def evaluate_model(
    model: nn.Module, inputs: torch.Tensor, labels: torch.Tensor
) -> dict[str, float | int]:
    if labels.ndim != 1 or labels.shape[0] != inputs.shape[0]:
        raise ValueError("labels must be rank 1 with the same batch size as inputs")
    model.eval()
    with torch.no_grad():
        # Local contract
        # - model.eval()과 no_grad 문맥에서는 학습용 동작과 gradient 기록을 끕니다.
        # - logits shape는 (batch, classes), labels shape는 (batch,)입니다.
        # - argmax는 class 축에서 적용해 shape (batch,)의 prediction을 만듭니다.
        # - 정확도는 맞은 비율을 뜻하는 Python float, count는 예시 수를 뜻하는 Python int입니다.
        # - 반환 schema는 accuracy와 count 두 key를 가진 dict입니다.
        # 학습자 편집 구간 시작 — 경계 주석은 남겨 두세요.
        # TODO: logits, argmax prediction, accuracy, count를 계산해 반환 dict를 완성하세요.'
        logits = model(inputs)
        predictions = logits.argmax(dim=1)

        accuracy = (predictions == labels).float().mean().item()
        result = {
            "accuracy": accuracy,
            "count": labels.shape[0]
        }


        # 학습자 편집 구간 끝

    return result


def build_comparison_models(hidden_size: int) -> dict[str, nn.Module]:
    return {
        "rnn": SequenceClassifier("rnn", input_size=1, hidden_size=hidden_size),
        "lstm": SequenceClassifier("lstm", input_size=1, hidden_size=hidden_size),
        "baseline": LastTokenLinearBaseline(),
    }

# 실행 순서: 이 구현 셀 → 바로 아래 fixture → check_e07()


In [46]:
torch.manual_seed(13)
train_x, train_y = make_delayed_dataset(examples=128, delay=12, seed=13)
eval_x, eval_y = make_delayed_dataset(examples=128, delay=12, seed=14)
models = build_comparison_models(hidden_size=8)
comparison_history = {
    name: fit_model(model, train_x, train_y, epochs=80, learning_rate=0.02)
    for name, model in models.items()
}
comparison = {name: evaluate_model(model, eval_x, eval_y) for name, model in models.items()}
print("last train loss:", {name: history[-1] for name, history in comparison_history.items()})
print("held-out comparison:", comparison)


last train loss: {'rnn': 0.000621919403783977, 'lstm': 0.0024547160137444735, 'baseline': 0.6926590204238892}
held-out comparison: {'rnn': {'accuracy': 1.0, 'count': 128}, 'lstm': {'accuracy': 1.0, 'count': 128}, 'baseline': {'accuracy': 0.53125, 'count': 128}}


In [47]:
def check_e07() -> None:
    class FirstTokenOracle(nn.Module):
        def forward(self, inputs: torch.Tensor) -> torch.Tensor:
            first = inputs[:, 0, 0].long()
            return F.one_hot(first, num_classes=2).float() * 10.0

    inputs, labels = make_delayed_dataset(examples=5, delay=2, seed=15)
    result = evaluate_model(FirstTokenOracle(), inputs, labels)
    torch.testing.assert_close(torch.tensor(result["accuracy"]), torch.tensor(1.0))
    one_result = evaluate_model(FirstTokenOracle(), inputs[:1], labels[:1])
    torch.testing.assert_close(one_result["count"], 1)
    try:
        evaluate_model(FirstTokenOracle(), inputs, labels.unsqueeze(1))
    except ValueError:
        pass
    else:
        raise AssertionError("rank-2 labels must fail")


check_e07()


### 결과 해석

관찰한 train-loss와 held-out comparison을 바탕으로 RNN·LSTM·baseline의 결과를 적고, 매 step `detach`, 마지막 hidden이 아닌 잘못된 readout, logits/labels shape 불일치 중 하나가 왜 실패를 만드는지 진단해야 합니다.

RNN과 LSTM은 held-out accuracy 1.0을 기록했고, 마지막 토큰 baseline은 0.53125이다.

이 데이터의 마지막 토큰은 항상 0이므로, 마지막 hidden 대신 마지막 원본 토큰을 readout에 넣으면 첫 토큰의 단서를 사용할 수 없다.

다만 하나의 seed와 단순한 delay=12 synthetic dataset 결과이므로, 이것만으로 RNN과 LSTM의 일반적인 성능 우위를 결론 낼 수 는 없다.
